In [ ]:
print("Installing required libraries")

!pip install --no-cache-dir -q "numpy>=2.0.0" "pandas>=2.2.3" "torch>=2.3.0" \
    "torchvision>=0.18.0" "transformers>=4.42.4" "peft==0.11.1" "accelerate==0.30.1" \
    "trl==0.9.4" "datasets==2.19.2" "bitsandbytes==0.43.1" "numpy_financial"

#RESTART RUNTIME

In [ ]:
## EVALUATION SCRIPT - PHI-2 OPTIMIZED VERSION

import json
import os
import torch
import warnings
import textwrap
import ast
import re
import pandas as pd
import numpy as np
from numpy import ma
import pickle
import numpy_financial as npf
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [ ]:

HF_REPO_ID = "priyam-turakhia/phi-2-numpy-modernization-v1"

# Enhanced system prompt matching training format
SYSTEM_PROMPT = (
    "You are a Python code refactoring tool for NumPy. Your task is to replace only the deprecated functions in the given code snippet with their modern equivalents.\n"
    "Your response must be structured with two markdown sections:\n"
    "1. A '### Refactored Code' section containing ONLY the updated Python code block.\n"
    "2. A '### Deprecation Context' section containing a brief explanation of the deprecation.\n"
    "CRITICAL: Do NOT change the code's logic. Do NOT add imports. Do NOT add comments. Do NOT modify arguments or add methods like .astype(). "
    "If the code does not contain deprecated functions, return the original code and state that no changes were needed in the context section."
)

def build_user_prompt(sample, tokenizer):
    """Build Phi-2 format prompt (no ChatML needed)"""
    user_prompt = f"### INPUT CODE:\n```python\n{sample['input']}\n```"
    # Phi-2 simple format: Instruct: ... Output:
    full_prompt = f"Instruct: {SYSTEM_PROMPT}\n\n{user_prompt}\nOutput:"
    return full_prompt

def parse_model_output(raw_output, original_prompt):
    """Parse Phi-2 output (no ChatML tokens)"""
    # Remove the input prompt to get only the generated response
    if original_prompt in raw_output:
        output = raw_output.replace(original_prompt, "").strip()
    else:
        # Fallback: try to find where the output starts
        output = raw_output

    # Parse the structured response
    code_match = re.search(r"### Refactored Code\s*```python\n(.*?)\n```", output, re.DOTALL)
    context_match = re.search(r"### Deprecation Context\n(.*?)(?=\n###|\Z)", output, re.DOTALL)

    # Preserve indentation for code
    code = code_match.group(1) if code_match else ""
    context = context_match.group(1).strip() if context_match else ""

    if not code and not context:
        return output, "Context not found."
    return code, context

def parse_model_output_fallback(raw_output):
    """Fallback parser for alternative formats"""
    # Try alternative format with colons
    code_match = re.search(r"### Refactored Code:\s*```python\n(.*?)\n```", raw_output, re.DOTALL)
    context_match = re.search(r"### Deprecation Context:\n(.*?)(?=\n```|\Z)", raw_output, re.DOTALL)

    code = code_match.group(1) if code_match else ""
    context = context_match.group(1).strip() if context_match else ""

    if not code and not context:
        return raw_output, "Context not found."
    return code, context

#CHANGE FOR PHI-2 MODEL END

def clean_for_ast(code):
    return "\n".join(
        line.rstrip() for line in code.splitlines()
        if line.rstrip() and not line.strip().startswith("#")
    )

def compare_outputs(actual, expected):
    if isinstance(actual, np.ma.MaskedArray) and actual.ndim == 0: actual = actual.item() if not actual.mask else np.ma.masked
    if isinstance(expected, np.ma.MaskedArray) and expected.ndim == 0: expected = expected.item() if not expected.mask else np.ma.masked
    if actual is np.ma.masked and expected is np.ma.masked: return True
    if isinstance(expected, tuple) and isinstance(actual, tuple):
        if len(expected) != len(actual): return False
        return all(compare_outputs(a, e) for a, e in zip(actual, expected))
    if isinstance(actual, str) and isinstance(expected, str): return actual.replace(" ", "").replace("\n", "") == expected.replace(" ", "").replace("\n", "")
    is_actual_arraylike = isinstance(actual, (np.ndarray, list, tuple))
    is_expected_arraylike = isinstance(expected, (np.ndarray, list, tuple))
    if is_actual_arraylike and is_expected_arraylike:
        try:
            actual_arr, expected_arr = np.asarray(actual), np.asarray(expected)
            if any(np.issubdtype(arr.dtype, np.number) for arr in [actual_arr, expected_arr]): return np.allclose(actual_arr, expected_arr, equal_nan=True)
            return np.array_equal(actual_arr, expected_arr)
        except (ValueError, TypeError): return False
    if is_actual_arraylike != is_expected_arraylike: return False
    return actual == expected

def check_compiles(f, code):
    execution_scope = {}
    try:
        exec(code, EVAL_GLOBALS, execution_scope)
        compiled_function = execution_scope.get(f)
        if not callable(compiled_function):
            raise NameError(f"Function '{f}' was not defined correctly.")
        return True, "OK", compiled_function
    except Exception as e:
        return False, f"SyntaxError: {e}", None

def check_indentation(f, code):
    scope = {}
    try:
        exec(code, EVAL_GLOBALS, scope)
        scope.get(f)
        return True, "OK"
    except Exception as e:
        return False , f"IndentationError: {e}"

def check_no_deprecations(f, input):
    if not callable(f): return False, "Function not callable"
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        try:
            if isinstance(input, tuple): f(*input)
            else: f(input)
        except AttributeError as e:
            if "module 'numpy' has no attribute" in str(e):
                return False, "Output contained deprecated features"
        except Exception as e:
            return False, f"Error during execution, most likely due to deprecation: {e}"

        for item in w:
            if issubclass(item.category, DeprecationWarning):
                return False, "Output contained deprecated features"
    return True, "OK"

def check_functionality(fun, test_cases: list):
    for j, case in enumerate(test_cases):
        try:
            input = eval(case['input'], EVAL_GLOBALS)
            expected_output = eval(case['expected_output'], EVAL_GLOBALS)
            actual_output = fun(*input) if isinstance(input, tuple) else fun(input)
            if not compare_outputs(actual_output, expected_output):
                return False, f"Failed: [Expected: {repr(expected_output)}, Got: {repr(actual_output)}]"
        except Exception as e:
            return False, f"Error during execution of Test Case {j+1}: {e}"
    return True, "All Test Cases passed"

#evaluation
def main():
    print("Starting Phi-2 evaluation script...")

    print(f"Loading model from Hugging Face Hub: {HF_REPO_ID}")
    try:
        pipe = pipeline(
            "text-generation",
            model=HF_REPO_ID,
            device_map="auto",
            torch_dtype=torch.float16,  # Use float16 instead of bfloat16 for compatibility
            trust_remote_code=True
        )
        tokenizer = pipe.tokenizer
        print("Model loaded successfully!")
    except Exception as e:
        print(f"Failed to load model: {e}")
        return

    print(f"Loading validation data from: {VALIDATION_DATA_PATH}")

    if not os.path.exists(VALIDATION_DATA_PATH):
        print(f"Validation file not found! Please upload '{VALIDATION_DATA_PATH}'.")
        return

    with open(VALIDATION_DATA_PATH, 'r') as f:
        validation_data = json.load(f)

    results_list = []
    total_samples = len(validation_data)

    for i, sample in enumerate(validation_data):
        print(f"\nProcessing sample {i+1}/{total_samples}:")

        # Build Phi-2 format prompt
        prompt = build_user_prompt(sample, tokenizer)

        # Generate response with optimized settings
        raw_output = pipe(
            prompt,
            max_new_tokens=200,  # Reduced to prevent over-generation
            do_sample=False,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id
        )[0]['generated_text']

        # Parse the generated code
        generated_code, context = parse_model_output(raw_output, prompt)

        # Try fallback parser if primary fails
        if not generated_code:
            generated_code, context = parse_model_output_fallback(raw_output)

        print(f"Generated code:\n{generated_code}")

        # Uncomment for debugging/testing with expected output
        # generated_code = sample['output']

        if not generated_code:
            print("Model did not generate code. Skipping.")
            results_list.append({'sample_index': i, 'compiles': 'Fail: No code generated'})
            continue

        # Prepare code for execution
        dedented_code = textwrap.dedent(generated_code).strip()
        indented_code = textwrap.indent(dedented_code, "    ")
        full_code = sample['code_before'] + "\n" + indented_code + "\n" + sample['code_after']
        full_code_ni = sample['code_before'] + "\n" + generated_code + "\n" + sample['code_after']

        function_name = sample['code_before'].split('def ')[1].split('(')[0].strip()

        # Run all checks
        compiles, compiles_msg, compiled_function = check_compiles(function_name, full_code)
        indentation, indentation_msg = (check_indentation(function_name, full_code_ni) if compiles else (False, "Skipped"))

        if compiles:
            test_input = eval(sample['test_cases'][0]['input'], EVAL_GLOBALS)
            no_deprecations, no_deprecations_msg = check_no_deprecations(compiled_function, test_input)
        else:
            no_deprecations, no_deprecations_msg = False, "Skipped"

        if compiles and no_deprecations:
            functionality, functionality_msg = check_functionality(compiled_function, sample['test_cases'])
        else:
            functionality, functionality_msg = False, "Skipped"

        results_list.append({
            'sample_index': i,
            'compiles': 'Pass' if compiles else f'Fail: {compiles_msg}',
            'correct_indentation': 'Pass' if indentation else f'Fail: {indentation_msg}',
            'no_deprecations': 'Pass' if no_deprecations else f'Fail: {no_deprecations_msg}',
            'correct_functionality': 'Pass' if functionality else f'Fail: {functionality_msg}',
        })

    print("\nEvaluation COMPLETE!")

    # Generate results
    detailed_df = pd.DataFrame(results_list)
    detailed_df.to_csv(DETAILED_RESULTS_CSV, index=False)
    print("\n📋 Detailed Results Table:")
    print(detailed_df.to_string())

    # Calculate summary metrics
    metrics = {}
    metrics['total_samples'] = total_samples
    metrics['compiles'] = detailed_df['compiles'].str.startswith('Pass').sum()
    metrics['correct_indentation'] = detailed_df['correct_indentation'].str.startswith('Pass').sum()
    metrics['no_deprecations'] = detailed_df['no_deprecations'].str.startswith('Pass').sum()
    metrics['correct_functionality'] = detailed_df['correct_functionality'].str.startswith('Pass').sum()

    # Calculate weighted score
    summary_score = (
        metrics['compiles'] +
        metrics['correct_indentation'] +
        metrics['no_deprecations'] +
        (3 * metrics['correct_functionality'])
        # 3x weight on functionality
    )
    metrics['summary_score'] = summary_score

    summary_df = pd.DataFrame([metrics])
    summary_df.to_csv(SUMMARY_METRICS_CSV, index=False)
    print("\nSummary Metrics:")
    print(summary_df.to_string())

if __name__ == "__main__":
    main()